# 16a — MP-Declare Constraint Mining (RuM)

Mines MP-Declare constraints with data conditions from the BPIC17 event log using RuM's MINERful + MpEnhancer.
Categorizes into prefix-safe vs sequence constraints and saves to pkl for downstream notebooks.

In [1]:
import sys
import os
from pathlib import Path

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home'

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))

# src/ must be on sys.path for torch.load to unpickle event_log_loader classes
src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from src.interpretability.perturbation_methods import csv_to_xes, discover_mpdeclare
from src.interpretability.perturbation_methods import mpdeclare_to_declare_constraints
from src.interpretability.perturbation_methods import DeclareConstraintChecker

/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/lark/utils.py:163: DeprecationWarning: module 'sre_parse' is deprecated
  import sre_parse
/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/lark/utils.py:164: DeprecationWarning: module 'sre_constants' is deprecated
  import sre_constants


In [2]:
# ===== CONFIGURATION =====

# Test mode: use small subset for quick testing (avoids Java OOM)
TEST_MODE = True
TEST_N_CASES = 1000      # Number of cases for XES constraint mining

# Declare mining hyperparameters
MIN_SUPPORT = 0.95                # Minimum support threshold for MINERful (0.0–1.0)
DATA_CONDITIONS = 'ACTIVATIONS'   # 'ACTIVATIONS', 'TARGETS', 'BOTH', or 'NONE'

In [3]:
import pandas as pd
import pm4py
from tqdm.auto import tqdm

csv_path = _current / 'data' / 'BPI_Challenge_2017.csv'

if TEST_MODE:
    xes_path = _current / 'data' / 'BPI_Challenge_2017_test_subset.xes'
else:
    xes_path = _current / 'data' / 'BPI_Challenge_2017.xes'

if not xes_path.exists():
    with tqdm(total=4, desc="CSV → XES conversion") as pbar:
        pbar.set_postfix_str("reading CSV...")
        df = pd.read_csv(csv_path)
        pbar.update(1)

        if TEST_MODE:
            subset_cases = df["case:concept:name"].unique()[:TEST_N_CASES]
            df = df[df["case:concept:name"].isin(subset_cases)]
            print(f"TEST MODE: Subsetting to {len(subset_cases)} cases ({len(df)} events)")

        pbar.set_postfix_str("cleaning NaN values...")
        df = df.dropna(axis=1, how='all')
        for col in df.select_dtypes(include='number').columns:
            df[col] = df[col].fillna(0)
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].fillna('')
        pbar.update(1)

        pbar.set_postfix_str("formatting dataframe...")
        df = pm4py.format_dataframe(df, case_id="case:concept:name", activity_key="concept:name", timestamp_key="time:timestamp")
        pbar.update(1)

        pbar.set_postfix_str("writing XES...")
        pm4py.write_xes(pm4py.convert_to_event_log(df), str(xes_path))
        pbar.update(1)
        pbar.set_postfix_str("done")
    print(f"Converted CSV to XES: {xes_path}")
else:
    print(f"XES file already exists: {xes_path}")

CSV → XES conversion:   0%|          | 0/4 [00:00<?, ?it/s]

TEST MODE: Subsetting to 1000 cases (39932 events)


/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/pm4py/utils.py:1000: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn(


exporting log, completed traces ::   0%|          | 0/1000 [00:00<?, ?it/s]

Converted CSV to XES: /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/data/BPI_Challenge_2017_test_subset.xes


In [4]:
# Start JVM with extra heap before discover_mpdeclare (avoids OOM on large logs)
import jpype
if not jpype.isJVMStarted():
    from src.interpretability.perturbation_methods.revised_plus.rum_mpdeclare import RUM_JAR
    jpype.startJVM(f"-Djava.class.path={RUM_JAR}", "-Djava.awt.headless=true", "-Xmx4g", convertStrings=True)
    __import__('jpype.imports')

mpdeclare_constraints = discover_mpdeclare(
    xes_path,
    min_support=MIN_SUPPORT,
    data_conditions=DATA_CONDITIONS,
)
print(f"Mined {len(mpdeclare_constraints)} MP-Declare constraints")

MP-Declare discovery:   0%|          | 0/3 [00:00<?, ?it/s]

log4j:WARN No appenders could be found for logger (minerful.miner.core.MinerFulKBCore).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||


Converting to RuM format:   0%|          | 0/805 [00:00<?, ?it/s]

[discover_mpdeclare] Skipping 1 template type(s) unsupported by MpEnhancer data conditions: {'Succession'}
2026-03-08 14:44:01,553 INFO    [main] task.discovery.mp_enhancer.MpEnhancer - MpEnhancer (1552870927) started at: 1772977441552
2026-03-08 14:44:01,556 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Number of constraints to process: 109
2026-03-08 14:44:01,557 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 1: Constraint(supp=1.0): Precedence[A_Incomplete, A_Accepted] | |
2026-03-08 14:44:01,576 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 2: Constraint(supp=1.0): Precedence[A_Validating, A_Accepted] | |
2026-03-08 14:44:01,592 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 3: Constraint(supp=1.0): Precedence[O_Cancelled, A_Accepted] | |
2026-03-08 14:44:01,601 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 4: Constraint(su

Mar 08, 2026 2:44:01 PM com.github.fommil.jni.JniNamer arch
Mar 08, 2026 2:44:01 PM com.github.fommil.netlib.ARPACK <clinit>
Mar 08, 2026 2:44:01 PM com.github.fommil.jni.JniNamer arch
Mar 08, 2026 2:44:01 PM com.github.fommil.netlib.ARPACK <clinit>


2026-03-08 14:44:02,352 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 45: Constraint(supp=1.0): Precedence[W_Assess potential fraud, A_Validating] | |
2026-03-08 14:44:02,374 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 46: Constraint(supp=1.0): Precedence[W_Call incomplete files, A_Validating] | |
2026-03-08 14:44:02,725 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 47: Constraint(supp=1.0): Precedence[A_Incomplete, O_Create Offer] | |
2026-03-08 14:44:02,749 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 48: Constraint(supp=1.0): Precedence[A_Validating, O_Create Offer] | |
2026-03-08 14:44:02,770 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 49: Constraint(supp=1.0): Precedence[O_Cancelled, O_Create Offer] | |
2026-03-08 14:44:02,790 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Process

Extracting results:   0%|          | 0/119 [00:00<?, ?it/s]

Mined 119 MP-Declare constraints


In [5]:
# Build activity vocabulary from dataset
import torch
from tqdm.auto import tqdm
from src.interpretability.utils.tensor_decoder import TensorDecoder

# src/ must be on sys.path for torch.load to unpickle event_log_loader classes
src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

with tqdm(total=2, desc="Loading dataset") as pbar:
    pbar.set_postfix_str("loading pkl...")
    data_path = _current / 'encoded_data' / 'BPIC_2017_all_5_test.pkl'
    full_dataset = torch.load(data_path, weights_only=False)
    pbar.update(1)

    pbar.set_postfix_str("building vocabulary...")
    decoder = TensorDecoder(full_dataset)

    ACTIVITY_FEATURE = 'concept:name'
    activity_idx_to_label = decoder.idx_to_label[ACTIVITY_FEATURE]
    max_idx = max(activity_idx_to_label.keys())
    activity_names = [activity_idx_to_label.get(i, f'<unk_{i}>') for i in range(max_idx + 1)]

    # Build name -> index mapping for conversion
    activity_name_to_idx = {name: i for i, name in enumerate(activity_names)}
    pbar.update(1)
    pbar.set_postfix_str("done")

print(f"Activity vocabulary ({len(activity_names)}):")
for i, name in enumerate(activity_names):
    print(f"  {i}: {name}")

Loading dataset:   0%|          | 0/2 [00:00<?, ?it/s]

Activity vocabulary (28):
  0: <pad>
  1: A_Accepted
  2: A_Cancelled
  3: A_Complete
  4: A_Concept
  5: A_Create Application
  6: A_Denied
  7: A_Incomplete
  8: A_Pending
  9: A_Submitted
  10: A_Validating
  11: EOS
  12: O_Accepted
  13: O_Cancelled
  14: O_Create Offer
  15: O_Created
  16: O_Refused
  17: O_Returned
  18: O_Sent (mail and online)
  19: O_Sent (online only)
  20: W_Assess potential fraud
  21: W_Call after offers
  22: W_Call incomplete files
  23: W_Complete application
  24: W_Handle leads
  25: W_Personal Loan collection
  26: W_Shortened completion 
  27: W_Validate application


In [6]:
# Convert MPDeclareConstraint -> DeclareConstraint
from tqdm.auto import tqdm

with tqdm(total=3, desc="Post-processing constraints") as pbar:
    pbar.set_postfix_str("converting to DeclareConstraint...")
    all_constraints, data_conditions = mpdeclare_to_declare_constraints(
        mpdeclare_constraints, activity_name_to_idx
    )
    pbar.update(1)

    pbar.set_postfix_str("categorizing...")
    prefix_safe_constraints = set(DeclareConstraintChecker.prefix_safe_constraints(all_constraints))
    sequence_constraints = all_constraints - prefix_safe_constraints
    pbar.update(1)

    pbar.set_postfix_str("done")
    pbar.update(1)

print(f"Total constraints:        {len(all_constraints)}")
print(f"Prefix-safe constraints:  {len(prefix_safe_constraints)}")
print(f"Sequence constraints:     {len(sequence_constraints)}")
print(f"Data conditions:          {len(data_conditions)}")

Post-processing constraints:   0%|          | 0/3 [00:00<?, ?it/s]

Total constraints:        119
Prefix-safe constraints:  101
Sequence constraints:     18
Data conditions:          46


In [7]:
# Display all three categories
print("=" * 80)
print("PREFIX-SAFE CONSTRAINTS (monotonic violations — reliable for prefix scoring)")
print("=" * 80)
for c in sorted(prefix_safe_constraints, key=lambda x: (x.template.value, x.activities)):
    print(f"  {c.format(activity_names)}")

print(f"\n{'=' * 80}")
print("SEQUENCE CONSTRAINTS (require full trace — used as validity gate)")
print("=" * 80)
for c in sorted(sequence_constraints, key=lambda x: (x.template.value, x.activities)):
    print(f"  {c.format(activity_names)}")

print(f"\n{'=' * 80}")
print("DATA CONDITIONS")
print("=" * 80)
if data_conditions:
    for dc, cond in sorted(data_conditions.items(), key=lambda x: (x[0].template.value, x[0].activities)):
        print(f"  {dc.format(activity_names)}")
        print(f"    Condition: {cond}")
else:
    print("  (none)")

PREFIX-SAFE CONSTRAINTS (monotonic violations — reliable for prefix scoring)
  absence(W_Assess potential fraud, n=1)
  init(A_Create Application)
  precedence(A_Accepted, A_Incomplete)
  precedence(A_Accepted, A_Validating)
  precedence(A_Accepted, O_Cancelled)
  precedence(A_Accepted, O_Refused)
  precedence(A_Accepted, O_Returned)
  precedence(A_Accepted, O_Sent (mail and online))
  precedence(A_Accepted, O_Sent (online only))
  precedence(A_Accepted, W_Assess potential fraud)
  precedence(A_Accepted, W_Call after offers)
  precedence(A_Accepted, W_Call incomplete files)
  precedence(A_Accepted, W_Validate application)
  precedence(A_Complete, A_Incomplete)
  precedence(A_Complete, A_Validating)
  precedence(A_Complete, O_Refused)
  precedence(A_Complete, O_Returned)
  precedence(A_Complete, W_Assess potential fraud)
  precedence(A_Complete, W_Call incomplete files)
  precedence(A_Complete, W_Validate application)
  precedence(A_Concept, A_Incomplete)
  precedence(A_Concept, A_Valid

In [8]:
# Save to pkl
import pickle

constraints_pkl = {
    'all': all_constraints,
    'prefix_safe': prefix_safe_constraints,
    'sequence': sequence_constraints,
    'data_conditions': data_conditions,
    'activity_names': activity_names,
}

pkl_name = 'bpic17_constraints_test.pkl' if TEST_MODE else 'bpic17_constraints.pkl'
pkl_path = _current / 'encoded_data' / pkl_name

with open(pkl_path, 'wb') as f:
    pickle.dump(constraints_pkl, f)

print(f"Saved constraints to {pkl_path}")
print(f"  all: {len(constraints_pkl['all'])} constraints")
print(f"  prefix_safe: {len(constraints_pkl['prefix_safe'])} constraints")
print(f"  sequence: {len(constraints_pkl['sequence'])} constraints")
print(f"  data_conditions: {len(constraints_pkl['data_conditions'])} entries")
print(f"  activity_names: {len(constraints_pkl['activity_names'])} activities")

Saved constraints to /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/encoded_data/bpic17_constraints_test.pkl
  all: 119 constraints
  prefix_safe: 101 constraints
  sequence: 18 constraints
  data_conditions: 46 entries
  activity_names: 28 activities
